In [1]:
from utils import *
import k3d
from scipy.spatial import distance
from sklearn.neighbors import KDTree
st = time.time()

In [2]:
path="../../DATA/registration_test/FB_100030____FB,2817673489_full/study_0c1e55cc/"
path_subfolder1=path+"MR7_bb3b1a0b"#"MR3_33a7c6a2"#"MR4_c61744e9"#"MR3_33a7c6a2"
path_subfolder2=path+"MR5_2e0c276a"
path_subfolder3=path+"MR3_33a7c6a2"

In [3]:
# read 2 series (sagittal / coronal / axial)
sag_image=serie_reader(path_subfolder1)
cor_image = serie_reader(path_subfolder2)
ax_image= serie_reader(path_subfolder3)

vType= sag_image.GetPixelIDTypeAsString()

In [4]:
# get spacing of each volume (dependent of orientation)
spacing_sag=get_spacing(sag_image)
spacing_cor=get_spacing(cor_image)
spacing_ax=get_spacing(ax_image)

print(spacing_sag, spacing_cor, spacing_ax)

[[0.4375    0.        0.       ]
 [0.        0.4375    0.       ]
 [0.        0.        3.3000002]] [[0.4375 0.     0.    ]
 [0.     0.4375 0.    ]
 [0.     0.     3.3   ]] [[0.4375 0.     0.    ]
 [0.     0.4375 0.    ]
 [0.     0.     3.6   ]]


In [5]:
# get direction cosin matrix of each volume (dependent of orientation)
dir_sag = get_direction(sag_image)
dir_cor = get_direction(cor_image)
dir_ax = get_direction(ax_image)

In [6]:
#dot product between spacing and Direct Cosine Matrix
A_sag=np.dot(dir_sag, spacing_sag)
A_cor=np.dot(dir_cor, spacing_cor)
A_ax=np.dot(dir_ax, spacing_ax)

In [7]:
#get origin from axial, sag, cor series
origin_sag= np.array(sag_image.GetOrigin())
origin_cor= np.array(cor_image.GetOrigin())
origin_ax = np.array(ax_image.GetOrigin())
print(origin_cor, origin_ax, origin_sag)

[  20.95518583 -102.72428033   48.84761293] [  29.43934254 -105.36338197  -81.23900868] [ 117.90852282 -103.56080763   53.28039366]


In [8]:
#get intensities from each series
m_sag=sitk.GetArrayFromImage(sag_image)
m_cor=sitk.GetArrayFromImage(cor_image)
m_ax=sitk.GetArrayFromImage(ax_image)

print(ax_image.GetSize(), m_ax.shape)

(320, 320, 31) (31, 320, 320)


In [9]:
# calcul physical coordinates of each volume
print("sag \n")
xspa_sag=calcul_physicsCoo(origin_sag, A_sag, m_sag)
print("cor \n")
xspa_cor=calcul_physicsCoo(origin_cor, A_cor, m_cor)
print("ax \n")
xspa_ax=calcul_physicsCoo(origin_ax, A_ax, m_ax)

sag 



100%|██████████| 32/32 [00:06<00:00,  4.77it/s]


cor 



100%|██████████| 34/34 [00:06<00:00,  4.97it/s]


ax 



100%|██████████| 31/31 [00:06<00:00,  4.84it/s]


In [10]:
#concat all points from each volume
sag_d= pd.DataFrame(xspa_sag, columns=('x', 'y', 'z', 'i'))
cor_d= pd.DataFrame(xspa_cor, columns=('x', 'y', 'z', 'i'))
ax_d= pd.DataFrame(xspa_ax, columns=('x', 'y', 'z', 'i'))

datas = [sag_d, cor_d, ax_d] #sag_d, 

datas_combined = pd.concat(datas)
print(datas_combined.head())
coo_combined=datas_combined[['x', 'y', 'z']]
intensity_combined=datas_combined[['i']]
print(coo_combined.head(), intensity_combined.head())

newres=0.4#np.min(np.asarray([sag_image.GetSpacing(), cor_image.GetSpacing(), ax_image.GetSpacing()]))
print(newres)

            x           y          z    i
0  117.908523 -103.560808  53.280394  0.0
1  117.969399 -103.127564  53.280394  0.0
2  118.030276 -102.694320  53.280394  0.0
3  118.091152 -102.261076  53.280394  0.0
4  118.152028 -101.827832  53.280394  0.0
            x           y          z
0  117.908523 -103.560808  53.280394
1  117.969399 -103.127564  53.280394
2  118.030276 -102.694320  53.280394
3  118.091152 -102.261076  53.280394
4  118.152028 -101.827832  53.280394      i
0  0.0
1  0.0
2  0.0
3  0.0
4  0.0
0.5


In [11]:
#determine min coordinates, max coordinates

min_xyz=np.array([coo_combined['x'].min(), coo_combined['y'].min(), coo_combined['z'].min()]) 
max_xyz=np.array([coo_combined['x'].max(), coo_combined['y'].max(), coo_combined['z'].max()])
print(min_xyz.dtype, max_xyz)
new_origin, new_spacing, dir_new = define_param(newres, min_xyz, max_xyz)

sizeX=ceil((coo_combined['x'].max()-coo_combined['x'].min())/newres)
sizeY=ceil((coo_combined['y'].max()-coo_combined['y'].min())/newres)
sizeZ=ceil((coo_combined['z'].max()-coo_combined['z'].min())/newres)
print(newres, new_origin, new_spacing, dir_new, sizeX, sizeY, sizeZ)

float64 [164.71161767  62.8907225   56.71900472]
0.5 [-10.49432384  62.8907225  -90.56740399] [[0.5 0.  0. ]
 [0.  0.5 0. ]
 [0.  0.  0.5]] [[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]] 351 348 295


In [12]:
s=160 #crop 160

I=np.arange(int((sizeX/2)-s),int((sizeX/2)+s))
J=np.arange(int((sizeY/2)-s),int((sizeY/2)+s))
K=np.arange(int((sizeZ/2)-s),int((sizeZ/2)+s))
#np.dot(A_new, np.array([i,j,k]))
#print(I)
A_new=np.dot(dir_new, new_spacing)

ii, jj, kk = np.meshgrid(I,J,K, indexing='ij')
I012 = np.zeros((ii.shape[0], jj.shape[1], kk.shape[2], 3))
I012[:,:,:,0] = ii
I012[:,:,:,1] = jj
I012[:,:,:,2] = kk
#print(ii.shape, jj.shape, kk.shape)
#print(sizeX, sizeY, sizeZ)



In [13]:
XYZ=new_origin + np.einsum('ijkl,lm->ijkm', I012, A_new)
newpoints=np.reshape(XYZ, (-1,3))

#combined_points=coo_combined
#allint=intensity_combined.to_numpy().ravel()
#print(newpoints.shape, combined_points.shape , allint.shape)


In [14]:
print(sag_d.head())
sag_p=sag_d[['x', 'y', 'z']].to_numpy()
cor_p=cor_d[['x', 'y', 'z']].to_numpy()
ax_p=ax_d[['x', 'y', 'z']].to_numpy()

sag_i=sag_d[['i']].to_numpy().ravel()
cor_i=cor_d[['i']].to_numpy().ravel()
ax_i=ax_d[['i']].to_numpy().ravel()

            x           y          z    i
0  117.908523 -103.560808  53.280394  0.0
1  117.969399 -103.127564  53.280394  0.0
2  118.030276 -102.694320  53.280394  0.0
3  118.091152 -102.261076  53.280394  0.0
4  118.152028 -101.827832  53.280394  0.0


In [15]:
#tree = KDTree(combined_points)
#nearest_dist, nearest_ind = tree.query(newpoints, k=3)  # k=2 nearest neighbors where k1 = identity

tree_sag = KDTree(sag_p)
nearest_dist_sag, nearest_ind_sag = tree_sag.query(newpoints, k=2)  # k=2 nearest neighbors where k1 = identity

tree_cor = KDTree(cor_p)
nearest_dist_cor, nearest_ind_cor = tree_cor.query(newpoints, k=2)  # k=2 nearest neighbors where k1 = identity

tree_ax = KDTree(ax_p)
nearest_dist_ax, nearest_ind_ax = tree_ax.query(newpoints, k=2)  # k=2 nearest neighbors where k1 = identity



In [16]:
#print(sag_p[nearest_ind_sag[(len(nearest_ind_sag)//2)-1:(len(nearest_ind_sag)//2)+1]])

In [17]:
print(nearest_dist_sag[(len(nearest_dist_sag)//2)-5:(len(nearest_dist_sag)//2)+5])

[[0.57981847 0.7024811 ]
 [0.56319136 0.69627783]
 [0.55356099 0.65039584]
 [0.55129415 0.60501369]
 [0.55648083 0.5633104 ]
 [1.33681583 1.34482028]
 [1.35055113 1.36271494]
 [1.3592959  1.38584817]
 [1.37099692 1.39732685]
 [1.38557928 1.41163726]]


In [18]:
print(nearest_dist_cor[(len(nearest_dist_cor)//2)-5:(len(nearest_dist_cor)//2)+5])

[[ 0.58107773  0.70485811]
 [ 0.5677669   0.67533746]
 [ 0.5619647   0.62872935]
 [ 0.56390298  0.58588195]
 [ 0.54767864  0.57350324]
 [13.59691489 13.59973999]
 [13.61601551 13.61684569]
 [13.63425034 13.63540976]
 [13.6519528  13.65458606]
 [13.66995191 13.67213528]]


In [19]:
print(nearest_dist_ax[(len(nearest_dist_ax)//2)-5:(len(nearest_dist_ax)//2)+5])

[[13.39720909 13.40044594]
 [13.89539982 13.89837656]
 [14.39371601 14.396036  ]
 [14.89214508 14.89385221]
 [15.39067607 15.39180999]
 [10.43483542 10.43597346]
 [ 9.93731726  9.93772505]
 [ 9.43880323  9.43982287]
 [ 8.94045517  8.94064006]
 [ 8.44155398  8.44230246]]


In [20]:
nearest_int_sag=sag_i[nearest_ind_sag]
nearest_int_cor=cor_i[nearest_ind_cor]
nearest_int_ax=ax_i[nearest_ind_ax]

In [21]:
nearest_int=np.concatenate((nearest_int_sag, nearest_int_cor, nearest_int_ax), axis=1)
nearest_dist=np.concatenate((nearest_dist_sag, nearest_dist_cor, nearest_dist_ax), axis=1)
#print(nearest_dist[(len(nearest_dist)//2)-5:(len(nearest_dist)//2)+5])




In [22]:
min_p=np.min(np.asarray(nearest_dist), axis=1)
print(min_p[(len(min_p)//2)-5:(len(min_p)//2)+5])
min_ind=np.argmin(nearest_dist, axis=1)
print(min_ind[(len(min_ind)//2)-5:(len(min_ind)//2)+5])
min_int= nearest_int[np.arange(nearest_int.shape[0]), np.argmin(nearest_dist, axis=1)]
print(min_int[(len(min_int)//2)-5:(len(min_int)//2)+5])

[0.57981847 0.56319136 0.55356099 0.55129415 0.54767864 1.33681583
 1.35055113 1.3592959  1.37099692 1.38557928]
[0 0 0 0 2 0 0 0 0 0]
[ 17.  15.   7.   8.   8. 316. 312. 286. 278. 291.]


In [23]:
sumdist=np.sum(nearest_dist, axis=1)
newintd=nearest_int*nearest_dist
newint=np.sum(newintd, axis=1)/sumdist #np.nanmean(newintd, axis=1) 
print(newint[(len(newint)//2)-5:(len(newint)//2)+5])

[  6.46256485  13.02642193  12.71167195  12.84442468  12.98493114
 228.06531507 217.89809051 217.29822861 225.10915238 217.13241558]


In [24]:
newint_mean=np.nanmean(nearest_int, axis=1)
print(newint_mean[(len(newint_mean)//2)-5:(len(newint_mean)//2)+5])

[ 12.83333333  13.33333333  10.5         11.5         12.83333333
 255.83333333 249.         240.16666667 243.33333333 241.        ]


In [49]:
p=1.5
inv_dist=(1./nearest_dist)**p
sumdist_inv=np.sum(inv_dist, axis=1)
newintd_inv=inv_dist*nearest_int
newint_inv=np.round(np.sum(newintd_inv, axis=1)/sumdist_inv) #np.nanmean(newintd, axis=1) 
print(newint_inv[(len(newint_inv)//2)-5:(len(newint_inv)//2)+5])


[ 16.  14.   9.  11.  13. 314. 311. 281. 276. 282.]


In [50]:
print(newint.shape, XYZ.shape)

new_vol=np.reshape(newint_inv, (s*2, s*2, s*2))

(16777216,) (256, 256, 256, 3)


In [51]:
print(ax_image.GetSpacing(), (newres, newres, newres) )
itk_view2 = itk_view_from_simpleitk(new_vol.astype(np.float32), (newres, newres, newres), dir_new, new_origin)
#itk_view = itk_view_from_simpleitk(m_ax, ax_image.GetSpacing(), dir_ax, origin_ax)
itkwidgets.view(itk_view2)

(0.4375, 0.4375, 3.5999999256962862) (0.5, 0.5, 0.5)
(256, 256, 256)
here
la


Viewer(geometries=[], gradient_opacity=0.22, point_sets=[], rendered_image=<itk.itkImagePython.itkImageF3; pro…

In [47]:
et = time.time()
# get the execution time
elapsed_time = (et - st)/60
print('Execution time:', elapsed_time, 'minutes')

Execution time: 33.23736506303151 minutes


In [48]:
# Converting back to SimpleITK (assumes we didn't move the image in space as we copy the information from the original)
filename='SUPERTESTVOL06_01INV04RES.nii.gz'
nv=new_vol.astype(np.uint16)
result_image = sitk.Image(list(nv.shape), sitk.sitkUInt16)
result_image = sitk.GetImageFromArray(nv)
result_image.SetSpacing((newres, newres, newres))
result_image.SetOrigin(new_origin)
result_image.SetDirection(tuple(dir_new.flatten()))
    
# write the image
sitk.WriteImage(result_image, filename)

In [ ]:
#nearest_int=allint[nearest_ind]
#sumdist=np.sum(nearest_dist, axis=1)
#newint=nearest_int*nearest_dist
#newint=np.nanmean(nearest_int, axis=1) #np.sum(newint, axis=1)/sumdist

#nearest_int=np.concatenate((nearest_int_sag, nearest_int_cor, nearest_int_ax), axis=1)
#nearest_dist=np.concatenate((nearest_dist_sag, nearest_dist_cor, nearest_dist_ax), axis=1)

#invprop=((sumdist.reshape(-1, 1)-nearest_dist)/sumdist.reshape(-1, 1))

tau=newres*4#450
expsquare=np.exp((nearest_dist**2)*tau)
#print( np.sum(expsquare[(len(expsquare)//2)-5:(len(expsquare)//2)+40], axis=1))#expsquare[(len(expsquare)//2)-5:(len(expsquare)//2)+40],
newintau=np.round(np.sum(nearest_int*expsquare, axis=1)/np.sum(expsquare, axis=1))
print(newintau[(len(newintau)//2)-5:(len(newintau)//2)+5])

#newint=nearest_int*nearest_dist

In [ ]:
print(newint[(len(newint)//2)-5:(len(newint)//2)+40])

In [ ]:
def distance(pt_1, pt_2):
    pt_1 = np.array((pt_1[0], pt_1[1], pt_1[2]))
    pt_2 = np.array((pt_2[0], pt_2[1], pt_2[2]))
    return np.linalg.norm(pt_1-pt_2)


def closest_node(node, points):
    pts_list = []
    dist = 9999999
    for p in points:
        if distance(node, p) <= dist:
            dist = distance(node, p)
            pts_list.append(p)
    return pts_list

sub=1
d=newres*2
cpt=0
for x in range(len(XYZ[0])): #range((len(XYZ[0])//2)-sub, (len(XYZ[0])//2)+sub):
    for y in range(len(XYZ[1])): #range((len(XYZ[1])//2)-sub, (len(XYZ[1])//2)+sub):
        for z in range(len(XYZ[2])): #in range((len(XYZ[2])//2)-sub, (len(XYZ[2])//2)+sub):
            current_point=XYZ[x, y, z]
            cpt+=1
            #subdata=datas_combined[(datas_combined['x']<current_point[0]+d) & (datas_combined['x']>current_point[0]-d)]
            #subdata=subdata[(subdata['y']<current_point[1]+d) & (subdata['y']>current_point[1]-d)]
            #subdata=subdata[(subdata['z']<current_point[2]+d) & (subdata['z']>current_point[2]-d)]
            #print(subdata)
            #print(current_point, "\n", closest_node(current_point, subdata.to_numpy())) 
print(cpt)
et = time.time()
# get the execution time
elapsed_time = (et - st)/60
print('Execution time:', elapsed_time, 'minutes')